# FaceFuse walkthrough

A guided tour of the quality-adaptive hybrid face recognition pipeline: detection,
enhancement, the two feature branches, the fusion rule, and the genetic search.

**Before you start**

1. `pip install -e .` from the repository root
2. `python scripts/download_weights.py`
3. Put a few images per person in `data/gallery/<person name>/` and held-out
   images in `data/probe/<person name>/`

Every path below comes from `facefuse.config`, so nothing is machine-specific.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

from facefuse import config

print("repository root :", config.ROOT)
print("detector weights:", config.WEIGHTS_PATH.name, config.WEIGHTS_PATH.exists())
print("gallery         :", config.GALLERY_DIR)
print("probe           :", config.PROBE_DIR)

## 1. Face detection

`FaceDetector` wraps a YOLO11-L checkpoint fine-tuned on WIDER FACE (single class:
`face`). Boxes are expanded by a 20% margin so the crop keeps forehead and chin
context, which the embedding model relies on.

In [ ]:
from facefuse.face_detection import FaceDetector, pad_image

detector = FaceDetector()          # uses config.WEIGHTS_PATH
image = cv2.imread(str(next(config.PROBE_DIR.rglob("*.jpg"))))

faces = detector.detect(image, extract_crops=True)
print(f"{len(faces)} face(s) detected")

preview = image.copy()
for face in faces:
    x1, y1, x2, y2 = face.bbox
    cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 200, 0), 2)

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("YOLO11-L face detections")
plt.show()

## 2. DIEM+ enhancement

A bilateral filter suppresses sensor noise while preserving the edges that both
feature branches depend on. Its three parameters are part of the genetic search
space, so the enhancer is configurable rather than fixed.

In [ ]:
from facefuse.enhancement import DIEMPlus

crop = max(faces, key=lambda f: f.confidence).crop
enhanced = DIEMPlus().process(crop)
padded = pad_image(enhanced)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [crop, enhanced, padded],
    ["raw crop", "DIEM+ enhanced", f"padded {padded.shape[0]}x{padded.shape[1]}"],
):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plt.show()

## 3. The two feature branches

* **Deep branch** - InsightFace `buffalo_l` produces a 512-d embedding.
* **Handcrafted branch** - a 256-bin LBP histogram concatenated with a 3780-d HOG
  descriptor. LBP compares each pixel to its eight neighbours, so it is invariant
  to any monotonic change in brightness; HOG encodes gradient orientation.

In [ ]:
from facefuse.hybrid_recognition import HandcraftedFeatureExtractor, get_image_quality
from facefuse.recognition import FaceRecognizer

recognizer = FaceRecognizer(device="auto")     # -1 -> CPU, 0 -> first GPU
handcrafted = HandcraftedFeatureExtractor()

embedding = recognizer.get_embedding(padded)
hand_feat = handcrafted.extract(enhanced)

print("deep embedding      :", embedding.shape)
print("handcrafted features:", hand_feat.shape, "(256 LBP + 3780 HOG)")
print(f"Laplacian variance  : {get_image_quality(enhanced):.1f}")

## 4. The fusion rule

The Laplacian variance of the enhanced crop is a cheap sharpness proxy. It sets
how much the deep score is trusted:

```
w_deep = clip(laplacian_variance / quality_scale, 0, 1)
w_hand = 1 - w_deep
score  = w_deep * cosine(deep) + w_hand * cosine(handcrafted)
```

A sharp frame is decided by the embedding. A blurred frame shifts weight onto the
illumination-invariant handcrafted descriptors instead of trusting a degraded
embedding.

In [ ]:
from facefuse.visualization import plot_quality_weighting

plot_quality_weighting(quality_scale=config.QUALITY_SCALE)

In [ ]:
# What blur does to the quality score, and therefore to the weights.
for ksize in (1, 5, 11, 21, 31):
    blurred = enhanced if ksize == 1 else cv2.GaussianBlur(enhanced, (ksize, ksize), 0)
    variance = get_image_quality(blurred)
    w_deep = min(1.0, variance / config.QUALITY_SCALE)
    print(f"blur k={ksize:>2}  lap_var={variance:8.1f}  "
          f"w_deep={w_deep:.2f}  w_hand={1 - w_deep:.2f}")

## 5. Enrolment and recognition

`HybridFaceDatabase` walks `data/gallery/<person>/` and stores an
`(embedding, handcrafted)` pair per image. `HybridFaceRecognizer` scores a probe
against every stored pair and returns the best fused match, or `Unknown` when
nothing clears the threshold.

In [ ]:
from facefuse.hybrid_recognition import HybridFaceDatabase, HybridFaceRecognizer

gallery = HybridFaceDatabase(recognizer, detector, DIEMPlus(), handcrafted)
gallery.build(str(config.GALLERY_DIR))

hybrid = HybridFaceRecognizer(gallery)
name, score = hybrid.identify(embedding, hand_feat, enhanced)
print(f"prediction: {name}  (fused score {score:.3f})")

## 6. Genetic hyper-parameter search

Five parameters interact non-linearly and there is no gradient to follow, so a
small real-valued GA (arithmetic crossover, Gaussian mutation, truncation
selection) searches them jointly.

The cell below optimises a toy objective so it runs in a second. The real search
over the recognition parameters is `facefuse benchmark`.

In [ ]:
from facefuse.genetic_optimizer import GeneticOptimizer

optimizer = GeneticOptimizer(population_size=30, generations=25, mutation_rate=0.3, seed=0)
best, fitness = optimizer.optimize(
    lambda ind: -((ind[0] - 3.0) ** 2) - ((ind[1] + 1.0) ** 2),
    bounds=[(-10.0, 10.0), (-10.0, 10.0)],
    verbose=False,
)
print(f"optimum near (3, -1): found {np.round(best, 3)} with fitness {fitness:.4f}")
print("\nSearch space used by the real pipeline:")
for (low, high), label in zip(
    config.GA_BOUNDS,
    ["threshold", "quality_scale", "bilateral d", "sigmaColor", "sigmaSpace"],
):
    print(f"  {label:<14} [{low}, {high}]")

## 7. Full benchmark

`facefuse benchmark` runs everything end to end: build both galleries, search the
parameters, evaluate hybrid against the deep-only baseline, and write the figures
and `metrics.json` to `outputs/`.

```bash
facefuse benchmark --no-show
```

Read the accuracy numbers with the caveat in the README's *Limitations* section:
the genetic search tunes on the same probe set it is scored on, so the tuned
figures are optimistic. Use `--no-optimize` for an untuned comparison.

In [ ]:
from facefuse.pipeline import main as run_benchmark

summary = run_benchmark(["--no-show", "--generations", "3", "--population", "6"])
summary["deep_only"], summary["hybrid"]